In [121]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException   
from datetime import datetime
import traceback

import pandas as pd
import re
import numpy as np
from time import sleep
import urllib.parse

pd.set_option('display.max_colwidth', None)

---
### Explorer crawler

---

In [122]:
# Set up functions
def print_update(func):
    def wrapper(*args, **kwargs):
        print(f'Running {func.__name__}')
        return func(*args, **kwargs)
    return wrapper


def reject_cookies(driver, temp=0, retries=2):
    try:
        WebDriverWait(driver, retries).until(EC.element_to_be_clickable((By.ID, 'onetrust-reject-all-handler')))
        driver.find_element(By.ID, 'onetrust-reject-all-handler').click()
    except (NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException):
        if temp >= retries:
            print(f'{retries} attempts failed to Reject cookies')
        else:
            reject_cookies(driver, temp + 1)


def skip_tutorial(driver, temp=0, retries=2):
    try:
        WebDriverWait(driver, retries).until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'walla-button.TooltipWrapper__skip')))
        driver.find_element(By.CSS_SELECTOR, 'walla-button.TooltipWrapper__skip').click()
        driver.find_element(By.CSS_SELECTOR, 'walla-button.TooltipWrapper__skip').click()
        driver.find_element(By.CSS_SELECTOR, 'walla-button.TooltipWrapper__skip').click()
    except (NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException):
        if temp >= retries:
            print(f'{temp} attempts failed to Skip the tutorial')
        else:
            skip_tutorial(driver, temp + 1)

@print_update
def setup_driver():
    options = webdriver.ChromeOptions()
    prefs = {"profile.managed_default_content_settings.images": 2}
    options.add_experimental_option("prefs", prefs)
    options.add_argument("--headless")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--log-level=4")
    options.add_argument("--disable-extensions")
    driver = webdriver.Chrome(options=options)
    driver.maximize_window()

    driver.get('https://es.wallapop.com/app/search?filters_source=category_navigation&latitude=40.41956&longitude=-3.69196')

    reject_cookies(driver)
    skip_tutorial(driver)

    return driver

@print_update
def get_explorer_urls(product_id=None, category_id=24200):
    # gets the corresponding urls to wallapop listing explorer
    base = 'https://es.wallapop.com/app/search?'
    urls = []

    if product_id:
        pass ## Do later
        # url = url if (product_id == None) else url + f'keywords={urllib.parse.quote(product_id)}&'
    else:
        '''
        Make a df for every category 
        Get the histogram curve for each category type by querying the listings db
        '''


        if category_id == 24200:
            object_type_ids = [9447, 9487, 10175, 24102]
            object_type_ids = [9447]
            for object_id in object_type_ids:
                url = base + f'category_ids={category_id}&filters_source=quick_filters&latitude=40.41956&longitude=-3.69196&object_type_ids={object_id}&'
                urls += split_explorer_url(url, product_id=product_id, category_id=category_id, object_id=object_id)
        else:
            print('category not supported')

    return urls


def split_explorer_url(url, product_id=None, category_id=None, object_id=None):
    # breaks up url into smaller subsets to divide up scrape load
    urls = []

    if product_id:
        # IGNORE write later after figuring out product_id classification module
        pass
    else:
        start, increment, loops = 0, 4, 12
        current_range = increment

        for i in range(loops):
            min = start
            max = start + current_range

            if i == 0:
                pricequery = f'max_sale_price={max}'
            elif i + 1 == len(range(loops)):
                pricequery = f'min_sale_price={min+.01}'
            else:
                pricequery = f'min_sale_price={min+.01}&max_sale_price={max}'

            start += current_range
            current_range += increment + (i-1)**2 # Ramps up range ~according to previous scrapes

            urls.append({
                'explorer_url': url + pricequery,
                'product_id': product_id,
                'category_id': category_id,
                'object_id': object_id,
                'date_scraped': np.nan,
                'scrape_empty': np.nan,
                'scrape_incomplete': np.nan,
                'redundant_url': np.nan,
            })
    return urls


def load_more(driver, temp=0, retries=2):
    # Clicks button that initiates infinite scroll dynamic listings
    try:
        WebDriverWait(driver, retries).until(EC.element_to_be_clickable((By.ID, 'btn-load-more')))
        driver.find_element(By.ID, 'btn-load-more').click()
    except (NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException):
        if temp >= retries:
            print(f'{temp} attempts failed to Click the load button')
        else:
            load_more(driver, temp + 1)


def wait_listings(driver, temp=0, retries=3):
    try:
        WebDriverWait(driver, retries).until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'a.ItemCardList__item')))
        return True
    except (NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException):
        if temp >= retries:
            print(f'{temp} attempts failed to Find listings')
        else:
            wait_listings(driver, temp + 1)

In [123]:
# Scraping functions
def scrape_explorer(driver, object_id=None):
    # Scrapes listings displayed in explorer
    scrapings = []
    cards = driver.find_elements(By.CSS_SELECTOR, 'a.ItemCardList__item')
    for card in cards:
        title = card.get_attribute('title')
        href = card.get_attribute('href')
        href = href.split('https://es.wallapop.com/item/')[-1]
        listing_id = href.split('-')[-1]
        num_images = len(card.find_elements(By.TAG_NAME, 'li'))
        # img_scr = card.find_element(By.TAG_NAME, 'img').get_attribute('src')
        date_scraped = datetime.now().strftime('%d%m%y')

        price = card.find_element(By.CSS_SELECTOR, 'span.ItemCard__price').text.strip()
        price = float(re.sub(r'[^\d,]', '', price).replace(',', '.'))
        price = int(price*100)
        
        try:
            if card.find_element(By.CSS_SELECTOR, 'p.ItemCard__highlight-text.pb-2'):
                featured = 1
            else:
                featured = 0
        except NoSuchElementException:
            featured = 0

        shadow_hosts = card.find_elements(By.CSS_SELECTOR, 'wallapop-badge')
        shipping, reserved, walla_pro = 0, 0, 0
        for shadow_host in shadow_hosts:
            shadow_root = driver.execute_script('return arguments[0].shadowRoot', shadow_host)
            span = shadow_root.find_element(By.CSS_SELECTOR, 'span').text.strip() 
            if span == 'Sólo venta en persona':
                shipping = 0
            elif span == 'Envío disponible':
                shipping = 1
            elif span == 'Envío gratis':
                shipping = 2
            elif span == 'Reservado':
                reserved = 1
            elif span == 'Reacondicionado':
                walla_pro = 1
            else:
                print('\nUNEXPECTED SHADOW_ROOT DATA\n')

        scrapings.append({
            'title': title,
            'https://es.wallapop.com/item/': href,
            'product_id': None,
            'object_id': object_id,
            'listing_id': listing_id,
            'num_images': num_images,
            # 'img_scr': img_scr,
            'date_last_scraped': date_scraped,
            'date_first_scraped': None,
            'price_cents': price,
            'featured': featured,
            'shipping': shipping,
            'reserved': reserved,
            'walla_pro': walla_pro,
        })
        
    return scrapings


def scroll_explorer(driver, object_id=None, max_scrolls=50):
    previous_height = driver.execute_script("return document.body.scrollHeight")
    sleep_time = 0.1
    count = 0
    timer = datetime.now()

    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        count += 1
        if count == max_scrolls:
            print('Scroll: Count reached')
            if check_scraping(driver, False, timer, object_id=object_id):
                return True

        try:
            WebDriverWait(driver, sleep_time).until(lambda driver: driver.execute_script("return document.body.scrollHeight") != previous_height)
            sleep_time /= 2
            previous_height = driver.execute_script("return document.body.scrollHeight")
        except TimeoutException:
            sleep_time += sleep_time
            if sleep_time > 12:
                print('Scroll: Timed out')
                if check_scraping(driver, True, timer, object_id=object_id):
                    return True
                else:
                    return False


def check_scraping(driver, bool, timer=None, object_id=None):
    global scraped

    if bool:
        pass
    else:
        sleep(1)
        
    # Get scrapings and check if they already exist in global
    scrapings = scrape_explorer(driver, object_id=object_id)
    if scrapings:
        temp = pd.DataFrame(scrapings)

        if scraped.empty:
            scraped = temp
        else:
            if temp['listing_id'].isin(scraped['listing_id']).any():
                if temp['listing_id'].isin(scraped['listing_id']).all():
                    print('    scroll completely redundant')
                    print(f'    redundant scrape = {len(scrapings)}')
                    return True
                else:
                    print('    server is repeating listings')
                    scraped = pd.concat([scraped, temp]).drop_duplicates()
            else:
                scraped = pd.concat([scraped, temp]).drop_duplicates()
    

        elem_count = len(scrapings)
        if not bool:
            clear_explorer(driver)

        # # Print stats
        timespent = datetime.now() - timer
        print(f'    current scrape = {elem_count}')
        print(f'    scrape time = {timespent}')
        print(f'    efficiency = {timespent/elem_count}')
        # print(f'total scraped = {len(scraped)}')
    else:
        timespent = datetime.now() - timer
        print(f'    current scrape = 0')
        print(f'    scrape time = {timespent}')
        print(f'    efficiency = {timespent} wasted')


    if bool:
        return True
    else:
        return scroll_explorer(driver, object_id=object_id)


def clear_explorer(driver):
    driver.execute_script("window.scrollTo(0, 0);")
    try:
        # Find elements and delete all siblings
        element = driver.find_element(By.CSS_SELECTOR, 'a.ItemCardList__item')
        parent = element.find_element(By.XPATH, '..')
        siblings = parent.find_elements(By.XPATH, '*')
        for sibling in siblings:
            driver.execute_script("arguments[0].remove();", sibling)
    except NoSuchElementException:
        print('    No listings found to clear to clear')


def crawl_explorer(custom_url=False, category_id=24200):
    global scraped

    if custom_url:
        urls = pd.DataFrame([{'explorer_url':custom_url, 'product_id':np.nan}])
        pending_urls = urls
    else:
        try:
            urls = pd.read_csv('data/urls.csv')
        except FileNotFoundError:
            urls = pd.DataFrame(get_explorer_urls(category_id=category_id))
        urls['date_scraped'] = pd.to_datetime(urls['date_scraped'], errors='coerce')
        current_week = datetime.now().isocalendar()[:2] # isocalendar() returns -> year, week, weekday


        # Rework to be last 7 days, not current week
        pending_urls = urls[urls['date_scraped'].apply(lambda x: x.isocalendar()[:2] if pd.notnull(x) else None) != current_week]

    times = []
    efficiencies = []
    for i, url in pending_urls['explorer_url'].items():
        print(f'\n=== URL_{i} - {url}')
        scraped = pd.DataFrame()
        complete = False
        time = datetime.now()

        driver = setup_driver()
        driver.get(url + '&order_by=price_low_to_high')
        if wait_listings(driver):
            try:
                load_more(driver)
                complete = scroll_explorer(driver, pending_urls.iloc[i]['object_id'])
                driver.quit()
            except Exception:
                print('\nLow-to-high error:')
                traceback.print_exc()
                driver.quit()
            print(f'{len(scraped)} low-to-high scraped')

            if not complete:
                driver = setup_driver()
                driver.get(url + '&order_by=price_high_to_low')
                if wait_listings(driver):
                    try:
                        load_more(driver)
                        complete = scroll_explorer(driver, pending_urls.iloc[i]['object_id'])
                        urls.at[i, 'scrape_incomplete'] = False
                        driver.quit()
                    except Exception:
                        print('\nHigh-to-low error:')
                        traceback.print_exc()
                        urls.at[i, 'scrape_incomplete'] = True
                        driver.quit()
                    print(f'{len(scraped)} high-to-low scraped')
                else:
                    driver.quit()
            else:
                urls.at[i, 'scrape_incomplete'] = False
        else:
            driver.quit()

        # Update files
        urls.at[i, 'date_scraped'] = datetime.now()
        if scraped.empty:
            urls.at[i, 'scrape_empty'] = True
            urls.to_csv('data/urls.csv', index=False)
        else: 
            urls.at[i, 'scrape_empty'] = False
            try:
                listings = pd.read_csv(f'data/listings/{urls.iloc[i]['object_id']}_{i}_listings.csv')
                if scraped.isin(listings.to_dict(orient='list')).all().all():
                    urls.at[i, 'redundant_url'] = True
                else:
                    urls.at[i, 'redundant_url'] = False
                    listings = pd.concat([listings, scraped]).drop_duplicates()
                    listings.to_csv(f'data/listings/{urls.iloc[i]['object_id']}_{i}_listings.csv', index=False)
            except FileNotFoundError:
                scraped.to_csv(f'data/listings/{urls.iloc[i]['object_id']}_{i}_listings.csv', index=False)
            urls.to_csv('data/urls.csv', index=False)

        timespent = datetime.now() - time
        times.append(timespent.total_seconds())
        len_scraped = len(scraped)
        efficiency = timespent / len_scraped
        efficiencies.append(efficiency.total_seconds())

        if complete:
            print(f'\nSCRAPE COMPLETE')
        else:
            print(f'\nSCRAPE INCOMPLETE')
        print(f'    Took: {timespent}')
        print(f'    Listings: {len_scraped}')
        print(f'    Efficency: {efficiency}')
        print(f'    Average time: {sum(times) / len(times)} sec')
        print(f'    Average efficiency: {sum(efficiencies) / len(efficiencies)} sec\n')

    print('All urls scraped')

In [124]:
scraped = []
crawl_explorer()


=== URL_14 - https://es.wallapop.com/app/search?category_ids=24200&filters_source=quick_filters&latitude=40.41956&longitude=-3.69196&object_type_ids=10175&min_sale_price=13.01&max_sale_price=26
Running setup_driver
2 attempts failed to Reject cookies
Scroll: Count reached
    current scrape = 880
    scrape time = 0:00:40.858232
    efficiency = 0:00:00.046430
Scroll: Count reached
    current scrape = 920
    scrape time = 0:00:42.664492
    efficiency = 0:00:00.046374
Scroll: Count reached
    current scrape = 959
    scrape time = 0:00:45.123716
    efficiency = 0:00:00.047053
Scroll: Count reached
    current scrape = 957
    scrape time = 0:00:46.238698
    efficiency = 0:00:00.048316
Scroll: Count reached
    current scrape = 880
    scrape time = 0:00:42.602063
    efficiency = 0:00:00.048411
Scroll: Count reached
    current scrape = 873
    scrape time = 0:00:42.409646
    efficiency = 0:00:00.048579
Scroll: Count reached
    current scrape = 958
    scrape time = 0:00:48.023

WebDriverException: Message: unknown error: net::ERR_CONNECTION_RESET
  (Session info: chrome-headless-shell=127.0.6533.89)
Stacktrace:
0   chromedriver                        0x0000000102b95088 cxxbridge1$str$ptr + 1887276
1   chromedriver                        0x0000000102b8d764 cxxbridge1$str$ptr + 1856264
2   chromedriver                        0x000000010279c82c cxxbridge1$string$len + 88524
3   chromedriver                        0x00000001027957e8 cxxbridge1$string$len + 59784
4   chromedriver                        0x0000000102788008 cxxbridge1$string$len + 4520
5   chromedriver                        0x0000000102789a10 cxxbridge1$string$len + 11184
6   chromedriver                        0x00000001027883e4 cxxbridge1$string$len + 5508
7   chromedriver                        0x0000000102787c30 cxxbridge1$string$len + 3536
8   chromedriver                        0x0000000102787b7c cxxbridge1$string$len + 3356
9   chromedriver                        0x0000000102785954 core::str:

---
### Product clasifier
---

## Phone categories
Samsung, Apple, Xiaomi, OPPO, Vivo, Huawei, LG, Lenovo, ZTE, SONY, RIM, HTC, Nokia, Transsion, Motorola, Tecno, Realme, Google
## Laptop categories
HP, Lenovo, Dell, Apple, Asus, Acer

## Hardware
Panasonic, Intel, 

In [ ]:
'''
It sounds like you're on the right track with normalizing the strings! Here are some additional steps you can take to further categorize and group similar book titles:

1. **Spell Correction**: Use a spell-checking library like `pyspellchecker` or `SymSpell` to correct common typos and misspellings.

2. **Remove Special Characters**: Strip out any special characters, punctuation, and numbers that are not part of the book title.

3. **Tokenization**: Break down the titles into individual words (tokens) and remove common stop words (like "and", "the", etc.).

4. **Stemming/Lemmatization**: Reduce words to their base or root form using libraries like `nltk` or `spaCy`.

5. **Fuzzy Matching**: Use fuzzy matching techniques (e.g., `fuzzywuzzy` library) to compare and group titles that are similar but not identical.

6. **Clustering**: Apply clustering algorithms (e.g., K-means, DBSCAN) to group similar titles together based on their tokenized and normalized forms.

7. **Manual Review**: After automated processing, a manual review might still be necessary to ensure accuracy, especially for edge cases.

Here's a sample Python code snippet to get you started with some of these steps:
'''


import re
from spellchecker import SpellChecker
from fuzzywuzzy import fuzz
from fuzzywuzzy import process

# Sample list of book titles
titles = ["HP and the sorcerer's stone", "Harry Potter and the Sorcerer's Stone", "Harry Pótter and the Sorcerer's Stone", ...]

# Normalize titles
def normalize_title(title):
    title = title.lower().strip()
    title = re.sub(r'[^\w\s]', '', title)  # Remove special characters
    title = re.sub(r'\d+', '', title)  # Remove numbers
    return title

# Spell correction
spell = SpellChecker()
def correct_spelling(title):
    corrected_title = " ".join([spell.correction(word) for word in title.split()])
    return corrected_title

# Apply normalization and spell correction
normalized_titles = [normalize_title(title) for title in titles]
corrected_titles = [correct_spelling(title) for title in normalized_titles]

# Fuzzy matching
def group_titles(titles):
    grouped_titles = {}
    for title in titles:
        match = process.extractOne(title, grouped_titles.keys(), scorer=fuzz.token_sort_ratio)
        if match and match[1] > 80:  # Adjust threshold as needed
            grouped_titles[match[0]].append(title)
        else:
            grouped_titles[title] = [title]
    return grouped_titles

grouped_titles = group_titles(corrected_titles)

# Print grouped titles
for key, group in grouped_titles.items():
    print(f"Group: {key}")
    for title in group:
        print(f" - {title}")
